##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Prompting Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Prompting.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook contains examples of how to write and run your first prompts with the Gemini API.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Prompting.ipynb).

In [ ]:
%pip install -U -q "google-genai>=2.9.0"  # 2.0 for Interactions API

Note: you may need to restart the kernel to use updated packages.


### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [8]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

Select the model you want to use in this guide:

In [10]:
MODEL_ID = "gemini-3.5-flash" # @param ["gemini-3.1-flash-lite", "gemini-2.5-flash", "gemini-3.5-flash", "gemini-2.5-pro", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

## Run your first prompt

Use `interactions.create` to generate responses to your prompts. You provide an `input` string, and access the response via `interaction.steps[-1].text`.

In [12]:
from IPython.display import Markdown, display

interaction = client.interactions.create(
    model=MODEL_ID,
    input="Give me python code to sort a list"
)

display(Markdown(interaction.steps[-1].content[0].text))

<IPython.core.display.Markdown object>


## Use images in your prompt

You can pass images to the model in multiple ways. The simplest is to pass an image URL directly:

In [ ]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": "Describe what you see in this image."},
        {
            "type": "image",
            "uri": "https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg",
        },
    ],
)

display(Markdown(interaction.steps[-1].content[0].text))

This image shows a whimsical, hand-drawn concept sketch for a product called a **"Jetpack Backpack."** 

The sketch is drawn in blue ink on lined notebook paper with a red left margin. It features a central drawing of a backpack with steam billowing out of two thrusters at the bottom, surrounded by handwritten annotations pointing to its various features.

Here are the details of the annotated features labeled in the sketch:

*   **Main Title:** "JETPACK BACKPACK" is written in all-caps and underlined at the top.
*   **Design & Comfort:**
    *   **"Fits 18\" Laptop"**: An arrow points to the top zippered compartment of the backpack.
    *   **"Padded Strap Support"**: An arrow points to the shaded shoulder straps.
    *   **"Lightweight, Looks Like a Normal Backpack"**: An arrow points to the main body of the pack to emphasize its stealthy design.
*   **Propulsion & Performance:**
    *   **"Retractable Boosters"**: An arrow points to the two small nozzles at the bottom of the bag.
    *   **"Steam-Powered, Green/Clean"**: An arrow points to the large, billowing clouds of steam drawn underneath the boosters.
*   **Power & Charging:**
    *   **"USB-C Charging"**: An arrow points to the lower side of the backpack.
    *   **"15-Min Battery Life"**: Written in the lower-left corner, indicating the limited duration of the jetpack's power source.

You can also download images and pass them as base64-encoded data. This is useful for local files or when you need to preprocess the image:

In [14]:
import urllib.request
urllib.request.urlretrieve(
    "https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg",
    "image.jpg"
)
print("Downloaded: image.jpg")

Downloaded: image.jpg


In [15]:
import base64
from IPython.display import Image, display

# Load and display the image
display(Image(filename='image.jpg'))

# Prepare base64 encoding for the API
with open('image.jpg', 'rb') as f:
    image_data = base64.b64encode(f.read()).decode('utf-8')

<IPython.core.display.Image object>


In [16]:
prompt = """
    This image contains a sketch of a potential product along with some notes.
    Given the product sketch, describe the product as thoroughly as possible based on what you
   see in the image, making sure to note all of the product features. Return output in json format:
   {description: description, features: [feature1, feature2, feature3, etc]}
"""

Then you can include the image in your prompt by passing a list of input items to `interactions.create`.

In [18]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {"type": "text", "text": prompt},
        {"type": "image", "data": image_data, "mime_type": "image/jpeg"},
    ],
)

print(interaction.steps[-1].content[0].text)

```json
{
  "description": "The Jetpack Backpack is a high-tech personal transportation device designed to blend in as a standard, lightweight everyday bag. It functions as a functional piece of luggage, capable of fitting an 18-inch laptop, while incorporating a steam-powered propulsion system that is advertised as green and clean. The backpack features a pair of retractable boosters at its base and is powered by a battery that provides 15 minutes of flight time. For user convenience and comfort, it includes padded strap support and a modern USB-C charging port.",
  "features": [
    "Fits 18\" laptop",
    "Padded strap support",
    "Lightweight construction",
    "Discreet design that looks like a normal backpack",
    "USB-C charging capabilities",
    "15-minute battery life",
    "Retractable boosters",
    "Steam-powered propulsion system",
    "Eco-friendly 'green/clean' operation"
  ]
}
```


## Multi-turn conversations

The Interactions API enables you to have freeform conversations across multiple turns.

Instead of managing a chat session, you simply chain interactions using `previous_interaction_id`. Each new interaction automatically gets the context from the previous ones.

In [20]:
turn_1 = client.interactions.create(
    model=MODEL_ID,
    input="In one sentence, explain how a computer works to a young child."
)

print(turn_1.steps[-1].content[0].text)

A computer is like a super-fast brain in a box that follows your instructions to help you play games, watch videos, and learn new things.


In [21]:
# The interaction ID lets us reference this turn later
print(f"Interaction ID: {turn_1.id}")

Interaction ID: v1_ChdQMVFBYXZpZktzYjFqckVQMnBQNzhBSRIXUDFRQWF2aWZLc2IxanJFUDJwUDc4QUk


With the Interactions API, conversation state is managed server-side. You can retrieve previous interactions if needed, but for basic multi-turn, you just pass the `previous_interaction_id`.

In [23]:
# Each interaction stores its own history via previous_interaction_id
print(f"Turn 1 text: {turn_1.steps[-1].content[0].text}")

Turn 1 text: A computer is like a super-fast brain in a box that follows your instructions to help you play games, watch videos, and learn new things.


You can keep sending messages to continue the conversation by referencing the previous turn's ID:

In [25]:
turn_2 = client.interactions.create(
    model=MODEL_ID,
    input="Okay, how about a more detailed explanation to a high schooler?",
    previous_interaction_id=turn_1.id,
)

print(turn_2.steps[-1].content[0].text)

To understand how a computer works at a high school level, it’s best to view it as a system that transforms **input** into **output** using a cycle of electronic logic.

Here is the breakdown of how that process actually happens:

### 1. The Language of Electricity (Binary)
At the lowest level, a computer doesn't "know" what a cat video or a math problem is. It consists of billions of microscopic switches called **transistors**. These switches can only be in two states: **On (1)** or **Off (0)**. This is called **binary**. Everything you see—colors, sounds, and text—is just a massive code made of these 1s and 0s.

### 2. The Brain (The CPU)
The Central Processing Unit (CPU) is the engine. It operates on a continuous loop called the **Instruction Cycle**:
*   **Fetch:** It grabs an instruction from the memory (like "add these two numbers").
*   **Decode:** It figures out what that instruction means.
*   **Execute:** It performs the calculation using its Arithmetic Logic Unit (ALU).
This

## Set the temperature

Every prompt you send to the model includes parameters that control how the model generates responses. Use a `generation_config` dictionary to set these, or omit it to use the defaults.

Temperature controls the degree of randomness in token selection. Use higher values for more creative responses, and lower values for more deterministic responses.

Note: Although you can set the `candidate_count` in the generation_config, 2.0 and later models will only return 1 result.

In [29]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input='Give me a numbered list of cat facts.',
    generation_config={
        "max_output_tokens": 2000,
        "temperature": 1.9,
        "stop_sequences": ['\n6'],  # Limit to 5 facts.
    },
)

display(Markdown(interaction.steps[-1].content[0].text))

<IPython.core.display.Markdown object>


## Learn more

There's lots more to learn!

* For more fun prompts, check out [Market a Jetpack](https://github.com/google-gemini/cookbook/blob/main/examples/Market_a_Jet_Backpack.ipynb).
* Check out the [safety quickstart](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Safety.ipynb) next to learn about the Gemini API's configurable safety settings, and what to do if your prompt is blocked.
* For lots more details on using the Python SDK, check out the [get started notebook](./Get_started.ipynb) or the [documentation's quickstart](https://ai.google.dev/tutorials/python_quickstart).